In [10]:
from itertools import combinations

In [11]:
sensitive_features = ["age", "gender", "race"]

res = []
for i in range(1, len(sensitive_features) + 1):
    c = list(combinations(sensitive_features, i))
    print(c)
    res.extend(c)

[('age',), ('gender',), ('race',)]
[('age', 'gender'), ('age', 'race'), ('gender', 'race')]
[('age', 'gender', 'race')]


In [12]:
res

[('age',),
 ('gender',),
 ('race',),
 ('age', 'gender'),
 ('age', 'race'),
 ('gender', 'race'),
 ('age', 'gender', 'race')]

In [30]:
joined = ["_".join(pairs) for pairs in res]

In [31]:
joined

['age',
 'gender',
 'race',
 'age_gender',
 'age_race',
 'gender_race',
 'age_gender_race']

## **Test Basic Fairness Profiler**

In [1]:
import sys
sys.path.append("../src")

In [2]:
from fairness import FairnessProfiler
from utils import load_data

In [3]:
df = load_data("../data/hcc_dataset.csv")

In [4]:
df

,gender,symptoms,alcohol,hbsag,hbeag,hbcab,hcvab,cirrhosis,endemic,smoking,...,alp,tp,creatinine,nodules,major_dim,dir_bil,iron,sat,ferritin,class
0,Male,No,Yes,No,No,No,No,Yes,No,Yes,...,150.0,7.1,0.70,1.0,3.5,0.5,NaN,NaN,NaN,Lives
1,Female,NaN,No,No,No,No,Yes,Yes,NaN,NaN,...,NaN,NaN,NaN,1.0,1.8,NaN,NaN,NaN,NaN,Lives
2,Male,No,Yes,Yes,No,Yes,No,Yes,No,Yes,...,109.0,7.0,2.10,5.0,13.0,0.1,28.0,6.0,16.0,Lives
3,Male,Yes,Yes,No,No,No,No,Yes,No,Yes,...,174.0,8.1,1.11,2.0,15.7,0.2,NaN,NaN,NaN,Dies
4,Male,Yes,Yes,Yes,No,Yes,No,Yes,No,Yes,...,109.0,6.9,1.80,1.0,9.0,NaN,59.0,15.0,22.0,Lives
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,Female,No,Yes,NaN,NaN,NaN,Yes,Yes,No,Yes,...,109.0,7.6,0.70,5.0,3.0,NaN,NaN,NaN,NaN,Lives
161,Female,Yes,No,NaN,NaN,NaN,NaN,Yes,No,No,...,280.0,6.7,0.70,1.0,2.2,2.3,NaN,NaN,NaN,Dies
162,Male,No,Yes,No,No,No,No,Yes,No,Yes,...,181.0,7.5,1.46,5.0,18.6,NaN,NaN,NaN,NaN,Lives
163,Male,No,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,...,170.0,8.4,0.74,5.0,18.0,NaN,NaN,NaN,NaN,Dies


In [5]:
fairness_profile = FairnessProfiler(df, 
                                    sensitive_features=["gender", "diabetes"],
                                    target="class")

In [6]:
fairness_profile.get_sensitive_features()

['gender', 'diabetes']

In [7]:
fairness_profile.get_sensitive_groups()

['gender', 'diabetes', 'gender_diabetes']

In [20]:
df.pivot_table(
    values="alcohol",
    index="class",
    columns="gender",
    aggfunc="count"
)

gender,Female,Male
class,,
Dies,11,52
Lives,21,81


In [18]:
df.groupby(["class", "gender", "alcohol"]).size().reset_index(name="count")

,class,gender,alcohol,count
0,Dies,Female,No,7
1,Dies,Female,Yes,4
2,Dies,Male,No,8
3,Dies,Male,Yes,44
4,Lives,Female,No,14
5,Lives,Female,Yes,7
6,Lives,Male,No,14
7,Lives,Male,Yes,67


In [29]:
df.pivot_table(
    index="class", 
    columns=["gender", "alcohol", "symptoms", "ps"],
    aggfunc="size"
)

gender   Female                                                            \
alcohol      No                                 Yes                         
symptoms     No    Yes                           No                         
ps       Active Active Ambulatory Restricted Active Ambulatory Restricted   
class                                                                       
Dies        1.0    1.0        3.0        2.0    1.0        NaN        NaN   
Lives       3.0    5.0        3.0        1.0    3.0        2.0        1.0   

gender                         Male  ...                               \
alcohol                          No  ...               Yes              
symptoms                 Yes     No  ...        Yes     No              
ps       Selfcare Ambulatory Active  ... Restricted Active Ambulatory   
class                                ...                                
Dies          1.0        2.0    1.0  ...        1.0    2.0        1.0   
Lives         NaN        1.0    6.0  ...        4.0   18.0        4.0   

gender                                                                       
alcohol                                                                      
symptoms                        Yes                                          
ps       Restricted Selfcare Active Ambulatory Disabled Restricted Selfcare  
class                                                                        
Dies            2.0      1.0    8.0        8.0      4.0        8.0      9.0  
Lives           3.0      1.0   14.0        6.0      NaN        6.0      4.0  

[2 rows x 24 columns]

In [26]:
import pandas as pd

In [41]:
pivot = pd.pivot_table(
    df,
    index="class",
    columns=["gender", "alcohol", "symptoms"],
    aggfunc="size",
    fill_value=0
)

In [42]:
pivot

gender   Female             Male            
alcohol      No     Yes       No     Yes    
symptoms     No Yes  No Yes   No Yes  No Yes
class                                       
Dies          1   6   2   2    3   5   6  37
Lives         3   9   6   1    6   4  26  30

In [46]:
pivot_rate = round(pivot / pivot.values.sum() * 100, 2)

In [47]:
pivot_rate

gender   Female                    Male                    
alcohol      No         Yes          No          Yes       
symptoms     No   Yes    No   Yes    No   Yes     No    Yes
class                                                      
Dies       0.68  4.08  1.36  1.36  2.04  3.40   4.08  25.17
Lives      2.04  6.12  4.08  0.68  4.08  2.72  17.69  20.41

In [49]:
pivot_rate.values.sum()

99.99000000000001

In [5]:
sensitive_groups = ['gender', 'diabetes', 'gender_diabetes_bla_ble_bli']

In [ ]:
cenas = {}
for g in sensitive_groups:
    cenas[g] = g.split("_")

In [7]:
cenas

{'gender': ['gender'],
 'diabetes': ['diabetes'],
 'gender_diabetes_bla_ble_bli': ['gender', 'diabetes', 'bla', 'ble', 'bli']}

### **Trying Target Rate by Group**

In [1]:
import sys
sys.path.append("../src")

In [2]:
from fairness import FairnessProfiler
from utils import load_data

In [3]:
df = load_data("../data/hcc_dataset.csv")

In [4]:
fairness_profile = FairnessProfiler(df, 
                                    sensitive_features=["gender", "alcohol", "endemic"],
                                    target="class")

In [5]:
fairness_profile.get_sensitive_features()

['gender', 'alcohol', 'endemic']

In [6]:
fairness_profile.get_sensitive_groups()

['gender',
 'alcohol',
 'endemic',
 'gender_alcohol',
 'gender_endemic',
 'alcohol_endemic',
 'gender_alcohol_endemic']

In [7]:
fairness_profile.target_rate_by_group_table(group="gender")

gender,Female,Male
class,,
Dies,6.67,31.52
Lives,12.73,49.09


In [8]:
fairness_profile.target_rate_by_group_table(group="gender", target="symptoms")

gender,Female,Male
symptoms,,
No,8.16,27.89
Yes,12.24,51.70


In [9]:
fp = FairnessProfiler(df, 
                      sensitive_features=["gender", "alcohol", "endemic"],
                    )

In [10]:
fp.target_rate_by_group_table(group="gender", target="alcohol")

gender,Female,Male
alcohol,,
No,12.73,13.33
Yes,6.67,67.27
